# Case Study 01 — Green Coverage Grid, Taipei 案例一：台北市綠覆率網格化
### Real dataset: Taipei street & park trees 真實資料集：臺北市行道樹及公園樹木分布圖

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/andrewwangarchnycu/gis-open-data-workshop-2026/blob/main/case-studies/01-green-coverage-grid/notebook.ipynb)

This is the **real-data version** of the tree-density analysis from [Lesson 05](../../lessons/05-computational-gis/) and [`notebooks/03_spatial_analysis.ipynb`](../../notebooks/03_spatial_analysis.ipynb) — same method, but on an actual open dataset instead of hand-made sample coordinates. It is also the recommended dataset for the [One Map Challenge](../../exercises/04_one_map_challenge/).
這是[課程 05](../../lessons/05-computational-gis/)與[`notebooks/03_spatial_analysis.ipynb`](../../notebooks/03_spatial_analysis.ipynb)中樹木密度分析的**真實資料版本**——方法相同，但套用在真實開放資料上，而非手造的範例座標。也是[一張地圖挑戰](../../exercises/04_one_map_challenge/)建議使用的資料集。

**Difficulty 難度**: beginner-friendly, no registration required 初學者友善，無需申請帳號。

## Step 0 — The research question 研究問題

**Question 問題**: *Where in the city does tree coverage cluster, and where is it sparse?* 城市中哪裡的樹木覆蓋較密集，哪裡較稀疏？

**Required variables 所需變數**: tree point locations across the study area 研究範圍內的樹木點位

**Real data source 真實資料來源**:

| Item 項目 | Value 內容 |
|---|---|
| Dataset 資料集 | 臺北市行道樹及公園樹木分布圖 (Taipei street & park trees) |
| Publisher 提供機關 | 臺北市工務局公園處 (Taipei Parks & Street Lights Office) |
| Portal 平台 | [data.taipei](https://data.taipei/dataset/detail?id=7a49d00c-a5ff-4a6b-be9e-aaa6dc1ff7e8) |
| Street-tree CSV 行道樹資料 | `https://tppkl.blob.core.windows.net/blobfs/TaipeiTree.csv` |
| License 授權 | Public / free 公開、免費 |
| Key columns 主要欄位 | `TreeID`, `Dist`(district 行政區), `TreeType`(species 樹種), `Diameter`, `TreeHeight`, `SurveyDate`, `TWD97X`, `TWD97Y` |
| Coordinate system 座標系統 | **TWD97 / TM2 zone 121 → EPSG:3826** (already meters — no reprojection needed for buffering) |

Note the CRS: this dataset ships in a **projected, meter-based** CRS already (unlike OSM/GPS data, which is usually EPSG:4326 lon/lat). That is itself a useful teaching point from [Lesson 02](../../lessons/02-space-to-data/) — always check *before* assuming.
請留意座標系統：此資料集本身就是**投影、公尺制**座標（不同於一般 GPS／OSM 常見的 EPSG:4326 經緯度）。這正好呼應[課程 02](../../lessons/02-space-to-data/)的重點——動手前務必先確認，不要假設。

In [ ]:
!pip install geopandas shapely matplotlib pandas contextily -q

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point, box
import matplotlib.pyplot as plt
import contextily as cx

print("geopandas", gpd.__version__)

> **Note on chart text 圖表文字說明**: Colab's default Matplotlib font can't render Chinese characters (they show as boxes □), so plot titles/legends below are kept in English. All explanations stay bilingual in the surrounding text. To enable Chinese in charts, see the font-setup snippet in [`resources/python/README.md`](../../resources/python/README.md).
> Colab 預設的 Matplotlib 字型無法顯示中文（會變成方框 □），因此下方圖表中的標題／圖例保持英文，但周圍說明文字維持雙語。若想讓圖表也能顯示中文，請參考 [`resources/python/README.md`](../../resources/python/README.md) 中的字型設定範例。

## Step 1 — Load 載入

Two options below. **Option A** (default) uses a small offline sample that mirrors the real dataset's exact schema and coordinate system, so the notebook runs instantly for everyone with no network dependency — consistent with the workshop's reproducibility rule in [Lesson 05](../../lessons/05-computational-gis/). **Option B** pulls the live CSV (works in Colab, which has full internet access; may be blocked in restricted sandboxes).

以下有兩個選項。**選項 A**（預設）使用一份小型離線範例，其欄位與座標系統與真實資料集完全一致，讓筆記本對所有人都能立即執行、不依賴網路——符合[課程 05](../../lessons/05-computational-gis/)的可重現性原則。**選項 B** 會抓取即時 CSV（在 Colab 中可正常運作；在受限的沙盒環境中可能無法連線）。

In [ ]:
# Option A — offline sample, real schema & CRS (default) 選項 A — 離線範例，欄位與座標系統皆與真實資料一致
# 36 illustrative points around a Da'an District block, TWD97 (EPSG:3826) meters,
# deliberately clustered so the grid analysis below produces a visible gradient.
# 36 個示意點位於大安區某街廓周邊，TWD97（EPSG:3826）公尺制座標，
# 刻意做出群聚分布，讓下方的網格分析能呈現明顯的密度梯度。
rng = np.random.default_rng(42)

cluster_centers = [(302450, 2770200), (302700, 2770400), (302300, 2770550)]  # denser park-adjacent clusters
sparse_center = (302900, 2770100)  # sparser roadside stretch

pts = []
for cx, cy in cluster_centers:
    pts += list(zip(rng.normal(cx, 40, 9), rng.normal(cy, 40, 9)))
pts += list(zip(rng.normal(sparse_center[0], 90, 9), rng.normal(sparse_center[1], 90, 9)))

species = ["樟樹", "台灣欒樹", "榕樹", "阿勃勒", "木棉"]
dist_names = ["大安區"]

trees = gpd.GeoDataFrame(
    {
        "TreeID": range(1, len(pts) + 1),
        "Dist": rng.choice(dist_names, len(pts)),
        "TreeType": rng.choice(species, len(pts)),
        "Diameter": rng.integers(15, 60, len(pts)),
        "TreeHeight": rng.integers(3, 15, len(pts)),
    },
    geometry=[Point(xy) for xy in pts],
    crs="EPSG:3826",
)
trees.head()

**Expected output 預期輸出**: a table with `TreeID`, `Dist`, `TreeType`, `Diameter`, `TreeHeight`, `geometry` — 36 rows, each geometry a `POINT (...)` in meters.
一張含 `TreeID`、`Dist`、`TreeType`、`Diameter`、`TreeHeight`、`geometry` 欄位的表格，共 36 列，每個幾何為公尺制的 `POINT (...)`。

In [ ]:
# Option B (optional) — live fetch of the real, full dataset 選項 B（選用）— 抓取真實完整資料集
# Requires internet access (works in Google Colab). Not run by default.
# 需要網路連線（在 Google Colab 中可正常運作）。預設不會自動執行。
FETCH_LIVE = False  # set True in Colab to try it 在 Colab 中改為 True 即可嘗試

if FETCH_LIVE:
    try:
        url = "https://tppkl.blob.core.windows.net/blobfs/TaipeiTree.csv"
        raw = pd.read_csv(url, encoding="utf-8-sig")
        real_trees = gpd.GeoDataFrame(
            raw,
            geometry=gpd.points_from_xy(raw["TWD97X"], raw["TWD97Y"]),
            crs="EPSG:3826",
        )
        print("Loaded", len(real_trees), "real trees from data.taipei")
        real_trees.head()
    except Exception as e:
        print("Live fetch failed (expected in a restricted sandbox) — use Option A above.")
        print("Error 錯誤:", e)

## Step 2 — Define the study area & build a fishnet grid 定義研究範圍並建立網格

**Why a grid, not administrative boundaries 為什麼用網格而非行政區界**: districts (`Dist`) vary hugely in size and shape, so comparing tree *counts* between them is meaningless. A regular grid gives every cell the same area, so density becomes directly comparable — the same reasoning as a raster in [Lesson 02](../../lessons/02-space-to-data/).
行政區（`Dist`）大小與形狀差異極大，直接比較樹木「數量」沒有意義。規則網格讓每一格面積相同，密度因此可以直接比較——這與[課程 02](../../lessons/02-space-to-data/)中網格資料（raster）的邏輯相同。

In [ ]:
def make_fishnet(gdf, cell_size):
    """Build a regular square grid covering gdf's bounding box.
    建立覆蓋 gdf 邊界範圍的規則方格網。
    """
    minx, miny, maxx, maxy = gdf.total_bounds
    minx -= cell_size; miny -= cell_size  # small buffer so edge trees aren't clipped
    cells = []
    x = minx
    while x < maxx + cell_size:
        y = miny
        while y < maxy + cell_size:
            cells.append(box(x, y, x + cell_size, y + cell_size))
            y += cell_size
        x += cell_size
    return gpd.GeoDataFrame({"cell_id": range(len(cells))}, geometry=cells, crs=gdf.crs)

CELL_SIZE_M = 100  # 100m x 100m grid cells 100 公尺 x 100 公尺網格
grid = make_fishnet(trees, CELL_SIZE_M)
print(f"{len(grid)} grid cells at {CELL_SIZE_M}m resolution")

## Step 3 — Spatial join & calculate 空間 Join 與計算

**Why 為什麼**: spatial join attaches each tree to the grid cell it falls inside — this is the same operation as [QGIS Spatial Join](../../qgis/02_analysis/spatial-join.md), just in code. Counting per cell and dividing by cell area gives a density metric comparable to [Lesson 06](../../lessons/06-spatial-insight/)'s "pattern → interpretation" chain.
空間 join 會把每棵樹指派到它所在的網格——這與 [QGIS 空間 Join](../../qgis/02_analysis/spatial-join.md)是同一種運算，只是改用程式碼執行。以每格樹木數除以面積得到密度指標，銜接[課程 06](../../lessons/06-spatial-insight/)「樣式→詮釋」的推論鏈。

In [ ]:
joined = gpd.sjoin(trees, grid, how="left", predicate="within")
tree_counts = joined.groupby("cell_id").size().rename("tree_count")

grid = grid.merge(tree_counts, on="cell_id", how="left")
grid["tree_count"] = grid["tree_count"].fillna(0)
grid["area_ha"] = grid.geometry.area / 10_000  # m2 -> hectares
grid["trees_per_ha"] = grid["tree_count"] / grid["area_ha"]

# Drop empty-of-any-signal cells for a cleaner map (keep only cells that overlap the data extent)
grid_active = grid[grid["tree_count"] > 0].copy()
grid_active[["cell_id", "tree_count", "trees_per_ha"]].sort_values("trees_per_ha", ascending=False).head()

**Expected output 預期輸出**: a table of grid cells sorted by `trees_per_ha` descending — the top rows are the densest 100m cells, matching the cluster centers built in Step 1.
依 `trees_per_ha` 降冪排序的網格表——最前面幾列即為樹木密度最高的 100 公尺網格，對應 Step 1 建立的群聚中心。

## Step 4 — Add a basemap 加入底圖

**Why 為什麼**: this grid is real Da'an District geometry — a basemap makes that legible immediately, instead of asking the reader to trust an axis of raw meter coordinates.
這個網格是真實的大安區幾何範圍——加上底圖能讓這點立即可辨識，而不用讓讀者自行從公尺座標軸猜測位置。

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
grid_active.plot(ax=ax, column="trees_per_ha", cmap="Greens", edgecolor="white",
                  linewidth=0.5, alpha=0.8, legend=True, legend_kwds={"label": "Trees per hectare"})
trees.plot(ax=ax, color="black", markersize=6, alpha=0.6)

# Basemap, reprojected to match our layers' CRS (EPSG:3826)
# 底圖，重新投影至與圖層相同的 CRS（EPSG:3826）
cx.add_basemap(ax, crs=grid_active.crs.to_string(), source=cx.providers.CartoDB.Positron)

ax.set_title("Green coverage grid — tree density (trees / ha)")
ax.set_axis_off()
plt.show()

## Research interpretation exercise 研究詮釋練習

Apply the What/Where/Why/So-what chain from [Lesson 06](../../lessons/06-spatial-insight/):

- **What? 是什麼？** _______ (e.g. a handful of 100m cells have far higher tree density than the rest)
- **Where? 在哪裡？** _______ (which cells / what's nearby — a park? a wide boulevard?)
- **Why? 為什麼？** _______ (planting history? adjacent land use? maintenance budget?)
- **So what? 所以呢？** _______ (what follow-up question or policy question does this raise?)

**Next steps 接下來**:
1. Change `CELL_SIZE_M` to 50 or 200 — how does the pattern change? This is the modifiable areal unit problem (MAUP), a core GIS caution. 修改 `CELL_SIZE_M` 為 50 或 200——樣式如何改變？這正是 GIS 中重要的「可調整面積單元問題」(MAUP)。
2. Set `FETCH_LIVE = True` in Step 1 (in Colab) to run this on the real, full-city dataset. 在 Colab 中將 Step 1 的 `FETCH_LIVE` 改為 `True`，即可套用在真實的全市資料上。
3. Take this result into [Lesson 07 — Research Map Design](../../lessons/07-research-map-design/) and the [One Map Challenge](../../exercises/04_one_map_challenge/).